In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dendropy

GLOBAL_SEED = 1
rng = np.random.default_rng(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

In [ ]:
CSV_PATH = Path('data/processed/arenavirus_hantavirus/pathogen_taxonomy_pcr.csv')
HOST_TREE_PATH = Path('data/processed/arenavirus_hantavirus/cophylogeny/arha_upham_hosts_hanta_arena_linked.nwk')
OUTPUT_DIR = Path('data/processed/arenavirus_hantavirus/cophylogeny/pd_saturation_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COL_VIRUS = 'assigned_name'
COL_HOST = None
COL_YEAR = None
COL_GROUP = None

HOST_LABEL_MAP_PATH = None

VIRUS_GROUP_MAP_PATH = None

DROP_VIRUS_NAMES = [
    'unclassified Orthohantavirus',
    'unclassified Mammarenavirus',
    'Hantaviridae',
    'Arenaviridae',
    'Unclassified Arenaviridae',
]

Y0 = None
Y0_ALT = None

R_PERM = 10000

N_EXEMPLARS = 6
MIN_EXEMPLAR_HOSTS = 3
MIN_EXEMPLAR_YEAR_BATCHES = 3

SAVE_FORMATS = ['png', 'pdf']

print('CSV_PATH:', CSV_PATH)
print('HOST_TREE_PATH:', HOST_TREE_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f'Required input file not found: {CSV_PATH}')

raw = pd.read_csv(CSV_PATH, low_memory=False)
print('Shape:', raw.shape)
print('Columns:')
for c in raw.columns:
    print(' -', c)

raw.head(3)

In [ ]:
# Standardize core columns: virus_id, host_id, year, optional group

def pick_col(df, aliases, override=None):
    if override is not None:
        if override not in df.columns:
            raise ValueError(f'Override column not found: {override}')
        return override
    lower = {c.lower(): c for c in df.columns}
    for a in aliases:
        if a in lower:
            return lower[a]
    return None

virus_aliases = ['virus_id', 'virus', 'assigned_name', 'pathogen', 'pathogen_name']
host_aliases = ['host_tip_label', 'host_id', 'host_species', 'host', 'species']
year_aliases = ['year', 'sample_year', 'collection_year', 'report_year', 'first_year', 'pub_year', 'publication_year', 'detection_year', 'date', 'sample_date', 'collection_date']
group_aliases = ['group', 'virus_group', 'assigned_family', 'family']

virus_col = pick_col(raw, virus_aliases, override=COL_VIRUS)
host_col = pick_col(raw, host_aliases, override=COL_HOST)
year_col = pick_col(raw, year_aliases, override=COL_YEAR)
group_col = pick_col(raw, group_aliases, override=COL_GROUP)

if virus_col is None:
    raise ValueError(
        f'Could not auto-detect virus column. Found columns: {list(raw.columns)}. '
        'Set COL_VIRUS manually in parameter cell.'
    )

print('Using columns (initial detection):')
print(' virus ->', virus_col)
print(' host  ->', host_col)
print(' year  ->', year_col)
print(' group ->', group_col)


def coerce_year(v):
    if pd.isna(v):
        return np.nan
    s = str(v)
    try:
        y = int(float(s))
        if 1800 <= y <= 2100:
            return y
    except Exception:
        pass
    import re
    hits = re.findall(r'(18\d{2}|19\d{2}|20\d{2}|2100)', s)
    if not hits:
        return np.nan
    ys = [int(h) for h in hits if 1800 <= int(h) <= 2100]
    return min(ys) if ys else np.nan

# Start with virus + optional direct host/year
std = pd.DataFrame({
    'virus_id': raw[virus_col].astype(str).str.strip(),
})

if host_col is not None:
    std['host_id'] = raw[host_col].astype(str).str.strip()
else:
    std['host_id'] = np.nan

if year_col is not None:
    std['year'] = raw[year_col].map(coerce_year)
else:
    std['year'] = np.nan

std['group_raw'] = raw[group_col].astype(str).str.strip() if group_col is not None else ''

# Fallback host mapping via host_record_id -> host_species from host.csv
base_dir = CSV_PATH.parent
host_csv = base_dir / 'host.csv'
if std['host_id'].isna().all() and 'host_record_id' in raw.columns and host_csv.exists():
    host_df = pd.read_csv(host_csv, low_memory=False)
    if 'host_record_id' in host_df.columns and 'host_species' in host_df.columns:
        host_map = host_df[['host_record_id', 'host_species']].copy()
        host_map['host_record_id'] = host_map['host_record_id'].astype(str).str.strip()
        host_map['host_species'] = host_map['host_species'].astype(str).str.strip()
        tmp = raw[['host_record_id']].copy()
        tmp['host_record_id'] = tmp['host_record_id'].astype(str).str.strip()
        tmp = tmp.merge(host_map, on='host_record_id', how='left')
        std['host_id'] = tmp['host_species'].replace({'': np.nan})
        print('Host fallback applied from host.csv using host_record_id.')

# Fallback year mapping (priority: direct year > sequence-derived > citation-derived)
if std['year'].isna().all():
    # 1) sequence.csv via pathogen_record_id (or study_id)
    seq_csv = base_dir / 'sequence.csv'
    seq_year = None
    if seq_csv.exists():
        seq = pd.read_csv(seq_csv, low_memory=False)
        candidate_cols = [c for c in ['collection_date', 'create_date', 'update_date'] if c in seq.columns]
        if candidate_cols:
            sy = pd.DataFrame()
            if 'pathogen_record_id' in seq.columns:
                sy['pathogen_record_id'] = seq['pathogen_record_id'].astype(str).str.strip()
            if 'study_id' in seq.columns:
                sy['study_id'] = seq['study_id'].astype(str).str.strip()

            # earliest available year across candidate date columns
            yy = []
            for c in candidate_cols:
                yy.append(seq[c].map(coerce_year))
            ymat = pd.concat(yy, axis=1)
            sy['year_seq'] = ymat.min(axis=1, skipna=True)

            if 'pathogen_record_id' in sy.columns and 'pathogen_record_id' in raw.columns:
                s1 = sy.dropna(subset=['year_seq']).groupby('pathogen_record_id', as_index=False)['year_seq'].min()
                t1 = raw[['pathogen_record_id']].copy()
                t1['pathogen_record_id'] = t1['pathogen_record_id'].astype(str).str.strip()
                seq_year = t1.merge(s1, on='pathogen_record_id', how='left')['year_seq']
                print('Year fallback candidate from sequence.csv via pathogen_record_id.')
            elif 'study_id' in sy.columns and 'study_id' in raw.columns:
                s2 = sy.dropna(subset=['year_seq']).groupby('study_id', as_index=False)['year_seq'].min()
                t2 = raw[['study_id']].copy()
                t2['study_id'] = t2['study_id'].astype(str).str.strip()
                seq_year = t2.merge(s2, on='study_id', how='left')['year_seq']
                print('Year fallback candidate from sequence.csv via study_id.')

    # 2) citations.csv via study_id publication_year
    cit_csv = base_dir / 'citations.csv'
    cit_year = None
    if cit_csv.exists() and 'study_id' in raw.columns:
        cit = pd.read_csv(cit_csv, low_memory=False)
        if 'study_id' in cit.columns and 'publication_year' in cit.columns:
            c = cit[['study_id', 'publication_year']].copy()
            c['study_id'] = c['study_id'].astype(str).str.strip()
            c['year_cit'] = c['publication_year'].map(coerce_year)
            c = c.dropna(subset=['year_cit']).groupby('study_id', as_index=False)['year_cit'].min()
            t = raw[['study_id']].copy()
            t['study_id'] = t['study_id'].astype(str).str.strip()
            cit_year = t.merge(c, on='study_id', how='left')['year_cit']
            print('Year fallback candidate from citations.csv via study_id.')

    # combine fallbacks: sequence preferred, then citation
    if seq_year is not None:
        std['year'] = seq_year
    if cit_year is not None:
        std['year'] = std['year'].fillna(cit_year)

# Final cleaning
std['virus_id'] = std['virus_id'].astype(str).str.strip()
std['virus_id'] = std['virus_id'].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

# Drop requested harmonized virus names (case-insensitive exact match)
if DROP_VIRUS_NAMES is not None and len(DROP_VIRUS_NAMES) > 0:
    drop_set = {str(v).strip().casefold() for v in DROP_VIRUS_NAMES}
    before_drop = len(std)
    std = std[~std['virus_id'].astype(str).str.strip().str.casefold().isin(drop_set)].copy()
    print('Dropped rows from excluded virus names:', before_drop - len(std))

std['host_id'] = std['host_id'].astype(str).str.strip()
std['host_id'] = std['host_id'].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
std = std.dropna(subset=['virus_id', 'host_id', 'year']).copy()
std['year'] = std['year'].astype(int)

# If cutoffs are unset, use full temporal span of cleaned data
if Y0 is None:
    Y0 = int(std['year'].min())
if Y0_ALT is None:
    Y0_ALT = int(std['year'].max())

# Infer Arena/Hanta group from available group-like column

def infer_group(g):
    g = str(g).strip().lower()
    if 'arena' in g:
        return 'Arena'
    if 'hanta' in g:
        return 'Hanta'
    return np.nan

std['group'] = std['group_raw'].map(infer_group)

# Optional external mapping fallback
if std['group'].isna().all() and VIRUS_GROUP_MAP_PATH is not None:
    map_df = pd.read_csv(VIRUS_GROUP_MAP_PATH, sep=None, engine='python')
    if map_df.shape[1] < 2:
        raise ValueError('VIRUS_GROUP_MAP_PATH must have at least two columns: virus_id, group')
    map_df = map_df.iloc[:, :2].copy()
    map_df.columns = ['virus_id', 'group_map']
    map_df['virus_id'] = map_df['virus_id'].astype(str).str.strip()
    map_df['group_map'] = map_df['group_map'].astype(str).str.strip()
    std = std.merge(map_df, on='virus_id', how='left')
    std['group'] = std['group'].fillna(std['group_map'])
    std = std.drop(columns=['group_map'])

print('Rows after cleaning:', len(std))
print('Unique viruses:', std['virus_id'].nunique())
print('Unique hosts:', std['host_id'].nunique())
print('Year range:', std['year'].min(), '-', std['year'].max())
print('Using cutoffs: Y0 =', Y0, '| Y0_ALT =', Y0_ALT)
if std['group'].notna().any():
    print('Group counts:')
    print(std['group'].value_counts(dropna=False))
else:
    print('No usable group labels found yet (group-level comparison cell will show mapping instructions).')

std.head(5)

In [ ]:
# First report year per virus-host pair and year-batched discovery table
vh_first = (
    std.groupby(['virus_id', 'host_id'], as_index=False)
       .agg(first_year=('year', 'min'),
            group=('group', lambda s: s.dropna().iloc[0] if s.dropna().size else np.nan))
)

v_year_discovery = (
    vh_first.groupby(['virus_id', 'first_year'], as_index=False)
            .agg(hosts=('host_id', lambda s: sorted(set(s))),
                 n_new_hosts=('host_id', lambda s: len(set(s))),
                 group=('group', lambda s: s.dropna().iloc[0] if s.dropna().size else np.nan))
            .sort_values(['virus_id', 'first_year'])
)

print('virus-host unique pairs:', len(vh_first))
print('year-batched rows:', len(v_year_discovery))
vh_first.head(5), v_year_discovery.head(5)

In [ ]:
# Host tree load, label matching, unmatched report
if not HOST_TREE_PATH.exists():
    raise FileNotFoundError(f'Host tree not found: {HOST_TREE_PATH}')

host_tree = dendropy.Tree.get(path=str(HOST_TREE_PATH), schema='newick', preserve_underscores=True)
if not host_tree.is_rooted:
    host_tree.reroot_at_midpoint(update_bipartitions=False)

host_tip_to_node = {}
for leaf in host_tree.leaf_node_iter():
    if leaf.taxon is None or leaf.taxon.label is None:
        raise ValueError('Found unlabeled host tip in host tree.')
    host_tip_to_node[leaf.taxon.label] = leaf
host_tips = set(host_tip_to_node.keys())

print('Host tree tips:', len(host_tips))

explicit_map = {}
if HOST_LABEL_MAP_PATH is not None:
    m = pd.read_csv(HOST_LABEL_MAP_PATH, sep=None, engine='python')
    if m.shape[1] < 2:
        raise ValueError('HOST_LABEL_MAP_PATH must have >=2 columns: raw_label, tree_label')
    m = m.iloc[:, :2].copy()
    m.columns = ['raw', 'tree']
    explicit_map = dict(zip(m['raw'].astype(str).str.strip(), m['tree'].astype(str).str.strip()))


def resolve_host_label(h):
    h = str(h).strip()
    if h in host_tips:
        return h
    if h in explicit_map and explicit_map[h] in host_tips:
        return explicit_map[h]
    c1 = h.replace(' ', '_')
    if c1 in host_tips:
        return c1
    c2 = h.replace('_', ' ')
    if c2 in host_tips:
        return c2
    return np.nan

vh_first['host_tip_resolved'] = vh_first['host_id'].map(resolve_host_label)

unmatched = vh_first[vh_first['host_tip_resolved'].isna()][['virus_id', 'host_id']].drop_duplicates()
unmatched_path = OUTPUT_DIR / 'unmatched_host_labels.tsv'
unmatched.to_csv(unmatched_path, sep='\t', index=False)

print('Unmatched host labels:', len(unmatched))
print('Unmatched report:', unmatched_path)
if len(unmatched) > 0:
    print(unmatched.head(10).to_string(index=False))
    print('WARNING: dropping unmatched pairs from downstream PD calculations.')

vh_first = vh_first.dropna(subset=['host_tip_resolved']).copy()
vh_first['host_tip_resolved'] = vh_first['host_tip_resolved'].astype(str)
print('Matched virus-host pairs retained:', len(vh_first))

In [ ]:
# Core PD functions and caches

def compute_node_depths(tree):
    depths = {tree.seed_node: 0.0}
    for node in tree.preorder_node_iter():
        if node is tree.seed_node:
            continue
        p = node.parent_node
        bl = 0.0 if node.edge.length is None else float(node.edge.length)
        depths[node] = depths[p] + bl
    return depths


def tree_total_pd(tree):
    total = 0.0
    for node in tree.preorder_node_iter():
        if node is tree.seed_node:
            continue
        total += 0.0 if node.edge.length is None else float(node.edge.length)
    return float(total)


def mrca_two(a, b, depths):
    na, nb = a, b
    da, db = depths[na], depths[nb]
    while da > db and na.parent_node is not None:
        na = na.parent_node
        da = depths[na]
    while db > da and nb.parent_node is not None:
        nb = nb.parent_node
        db = depths[nb]
    while na is not nb:
        if na.parent_node is None or nb.parent_node is None:
            return na if na.parent_node is None else nb
        na, nb = na.parent_node, nb.parent_node
    return na


def mrca_many(nodes, depths):
    if len(nodes) == 0:
        return None
    m = nodes[0]
    for n in nodes[1:]:
        m = mrca_two(m, n, depths)
    return m


# Initialize depth/tree-wide constants before any function calls that depend on them
HOST_DEPTHS = compute_node_depths(host_tree)
PD_TREE_TOTAL = tree_total_pd(host_tree)

pd_cache = {}

def faith_pd(taxa_set):
    # Faith's PD: sum of branch lengths in minimal subtree spanning taxa_set
    key = frozenset(taxa_set)
    if key in pd_cache:
        return pd_cache[key]

    if len(key) == 0:
        pd_cache[key] = np.nan
        return np.nan
    if len(key) == 1:
        pd_cache[key] = 0.0
        return 0.0

    nodes = [host_tip_to_node[t] for t in key]
    m = mrca_many(nodes, HOST_DEPTHS)
    seen_edges = set()
    total = 0.0

    for node in nodes:
        cur = node
        while cur is not m:
            e = cur.edge
            eid = id(e)
            if eid not in seen_edges:
                seen_edges.add(eid)
                total += 0.0 if e.length is None else float(e.length)
            if cur.parent_node is None:
                break
            cur = cur.parent_node

    pd_cache[key] = float(total)
    return float(total)


def patristic_distance(a_tip, b_tip):
    a = host_tip_to_node[a_tip]
    b = host_tip_to_node[b_tip]
    m = mrca_two(a, b, HOST_DEPTHS)
    return float(HOST_DEPTHS[a] + HOST_DEPTHS[b] - 2.0 * HOST_DEPTHS[m])


OBS_HOSTS = sorted(vh_first['host_tip_resolved'].unique())
HOST_INDEX = {h: i for i, h in enumerate(OBS_HOSTS)}
HOST_DIST = np.zeros((len(OBS_HOSTS), len(OBS_HOSTS)), dtype=float)
for i in range(len(OBS_HOSTS)):
    for j in range(i + 1, len(OBS_HOSTS)):
        d = patristic_distance(OBS_HOSTS[i], OBS_HOSTS[j])
        HOST_DIST[i, j] = d
        HOST_DIST[j, i] = d


def mpd_mntd(hosts):
    hosts = list(hosts)
    k = len(hosts)
    if k < 2:
        return np.nan, np.nan
    idx = np.array([HOST_INDEX[h] for h in hosts], dtype=int)
    sub = HOST_DIST[np.ix_(idx, idx)]
    iu = np.triu_indices(k, 1)
    mpd = float(np.mean(sub[iu]))
    s2 = sub.copy()
    np.fill_diagonal(s2, np.inf)
    mntd = float(np.mean(np.min(s2, axis=1)))
    return mpd, mntd


def pd_accumulation_by_year(virus_id, vh_df):
    sub = vh_df[vh_df['virus_id'] == virus_id][['host_tip_resolved', 'first_year']].drop_duplicates()
    if sub.empty:
        return pd.DataFrame(columns=['virus_id', 'year', 'n_new_hosts', 'n_hosts_cum', 'pd_cum', 'delta_pd', 'hosts_added'])

    by_year = (
        sub.groupby('first_year')['host_tip_resolved']
           .agg(lambda s: sorted(set(s)))
           .reset_index()
           .sort_values('first_year')
    )

    seen = set()
    prev_pd = 0.0
    rows = []
    for _, r in by_year.iterrows():
        y = int(r['first_year'])
        new_hosts = [h for h in r['host_tip_resolved'] if h not in seen]
        seen.update(new_hosts)
        pd_now = faith_pd(seen)
        d_pd = pd_now - prev_pd
        rows.append({
            'virus_id': virus_id,
            'year': y,
            'n_new_hosts': len(new_hosts),
            'n_hosts_cum': len(seen),
            'pd_cum': pd_now,
            'delta_pd': d_pd,
            'hosts_added': tuple(new_hosts),
        })
        prev_pd = pd_now

    return pd.DataFrame(rows)


def pd_accumulation_by_hosts(virus_id, vh_df):
    sub = vh_df[vh_df['virus_id'] == virus_id][['host_tip_resolved', 'first_year']].drop_duplicates()
    if sub.empty:
        return pd.DataFrame(columns=['virus_id', 'host_idx', 'host_tip', 'year', 'n_hosts_cum', 'pd_cum'])

    ordered = sub.sort_values(['first_year', 'host_tip_resolved'])
    seen = set()
    rows = []
    for i, (_, r) in enumerate(ordered.iterrows(), start=1):
        h = r['host_tip_resolved']
        y = int(r['first_year'])
        seen.add(h)
        rows.append({
            'virus_id': virus_id,
            'host_idx': i,
            'host_tip': h,
            'year': y,
            'n_hosts_cum': len(seen),
            'pd_cum': faith_pd(seen),
        })
    return pd.DataFrame(rows)


def late_phase_fraction(accum_year_df, y0):
    if accum_year_df.empty:
        return np.nan
    total = accum_year_df['delta_pd'].sum()
    if total <= 0:
        return np.nan
    late = accum_year_df.loc[accum_year_df['year'] >= y0, 'delta_pd'].sum()
    return float(late / total)


def permutation_pd_curves(hosts, r=500, seed=1):
    hosts = list(sorted(set(hosts)))
    k = len(hosts)
    if k == 0:
        return np.empty((0, 0), dtype=float)

    rng_local = np.random.default_rng(seed)
    curves = np.zeros((r, k), dtype=float)

    for i in range(r):
        order = rng_local.permutation(hosts)
        seen = set()
        for j, h in enumerate(order):
            seen.add(h)
            curves[i, j] = faith_pd(seen)

    return curves


PD_ASSOC_TOTAL = faith_pd(set(OBS_HOSTS))

print('Observed hosts (assoc ∩ tree):', len(OBS_HOSTS))
print('Host tree total PD:', round(PD_TREE_TOTAL, 3))
print('Observed-host total PD:', round(PD_ASSOC_TOTAL, 3))

In [ ]:
# Build per-virus accumulations and summary metrics
viruses = sorted(vh_first['virus_id'].unique())

accum_year_by_virus = {}
accum_host_by_virus = {}
summary_rows = []

virus_group = (
    vh_first[['virus_id', 'group']]
    .drop_duplicates()
    .groupby('virus_id')['group']
    .agg(lambda s: s.dropna().iloc[0] if s.dropna().size else np.nan)
    .to_dict()
)

for v in viruses:
    ay = pd_accumulation_by_year(v, vh_first)
    ah = pd_accumulation_by_hosts(v, vh_first)
    accum_year_by_virus[v] = ay
    accum_host_by_virus[v] = ah

    final_hosts = int(ay['n_hosts_cum'].max()) if not ay.empty else 0
    final_pd = float(ay['pd_cum'].iloc[-1]) if not ay.empty else np.nan
    mpd, mntd = mpd_mntd(vh_first.loc[vh_first['virus_id'] == v, 'host_tip_resolved'].unique())

    max_delta = float(ay['delta_pd'].max()) if not ay.empty else np.nan
    max_delta_frac = (max_delta / final_pd) if np.isfinite(max_delta) and np.isfinite(final_pd) and final_pd > 0 else np.nan

    summary_rows.append({
        'virus_id': v,
        'group': virus_group.get(v, np.nan),
        'n_hosts_final': final_hosts,
        'pd_final': final_pd,
        'pd_final_norm_tree': (final_pd / PD_TREE_TOTAL) if np.isfinite(final_pd) and PD_TREE_TOTAL > 0 else np.nan,
        'pd_final_norm_assoc': (final_pd / PD_ASSOC_TOTAL) if np.isfinite(final_pd) and PD_ASSOC_TOTAL > 0 else np.nan,
        'pd_per_host': (final_pd / final_hosts) if final_hosts > 0 and np.isfinite(final_pd) else np.nan,
        'mpd_hosts': mpd,
        'mntd_hosts': mntd,
        'max_delta_pd': max_delta,
        'max_delta_frac': max_delta_frac,
        'f_late_y0': late_phase_fraction(ay, Y0),
        'f_late_y0_alt': late_phase_fraction(ay, Y0_ALT),
        'n_year_batches': int(len(ay)),
    })

virus_summary = pd.DataFrame(summary_rows).sort_values('virus_id').reset_index(drop=True)
accum_year_long = pd.concat(accum_year_by_virus.values(), ignore_index=True) if accum_year_by_virus else pd.DataFrame()
accum_host_long = pd.concat(accum_host_by_virus.values(), ignore_index=True) if accum_host_by_virus else pd.DataFrame()

virus_summary.to_csv(OUTPUT_DIR / 'virus_summary.csv', index=False)
accum_year_long.to_csv(OUTPUT_DIR / 'per_virus_pd_by_year.csv', index=False)
accum_host_long.to_csv(OUTPUT_DIR / 'per_virus_pd_by_hosts.csv', index=False)

print('Saved summary/long tables to', OUTPUT_DIR)
virus_summary.head(10)

In [ ]:
try:
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    import plotly.express as px
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("Plotly is not installed in this kernel. Run: pip install plotly") from e

virus_ids = sorted(v for v in accum_year_by_virus.keys() if not accum_year_by_virus[v].empty)

palette = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
virus_color = {v: palette[i % len(palette)] for i, v in enumerate(virus_ids)}

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        'Per-virus cumulative PD vs year',
        'Per-virus cumulative PD vs cumulative hosts',
    ],
    horizontal_spacing=0.1,
)

for v in virus_ids:
    d = accum_year_by_virus[v]
    fig.add_trace(
        go.Scatter(
            x=d['year'],
            y=d['pd_cum'],
            mode='lines+markers',
            name=v,
            legendgroup=v,
            line=dict(color=virus_color[v], width=1.6),
            marker=dict(size=4),
            opacity=0.85,
            hovertemplate='Virus: %{fullData.name}<br>Year: %{x}<br>Cumulative PD: %{y:.4f}<extra></extra>',
            showlegend=True,
        ),
        row=1,
        col=1,
    )

for v, d in accum_host_by_virus.items():
    if d.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=d['n_hosts_cum'],
            y=d['pd_cum'],
            mode='lines+markers',
            name=v,
            legendgroup=v,
            line=dict(color=virus_color.get(v, '#555555'), width=1.6),
            marker=dict(size=4),
            opacity=0.85,
            hovertemplate='Virus: %{fullData.name}<br>Cumulative hosts: %{x}<br>Cumulative PD: %{y:.4f}<extra></extra>',
            showlegend=False,
        ),
        row=1,
        col=2,
    )

fig.update_xaxes(title_text='Year (first report; ties batched)', row=1, col=1)
fig.update_yaxes(title_text='Cumulative Faith PD', row=1, col=1)
fig.update_xaxes(title_text='Cumulative hosts discovered', row=1, col=2)
fig.update_yaxes(title_text='Cumulative Faith PD', row=1, col=2)

fig.update_layout(
    height=560,
    width=1300,
    title='All-virus cumulative PD curves (interactive)',
    legend=dict(
        title='Virus (assigned_name)',
        orientation='v',
        x=1.02,
        y=1.0,
        xanchor='left',
        yanchor='top',
        font=dict(size=10),
    ),
    margin=dict(l=50, r=360, t=70, b=50),
    template='plotly_white',
)

fig.show()

fig.write_html(OUTPUT_DIR / 'all_viruses_pd_curves_interactive.html', include_plotlyjs='cdn')

In [ ]:
# Objective exemplar selection (depth-aware for faceted panels)
vs = virus_summary.copy()
vs = vs[np.isfinite(vs['pd_final']) & (vs['n_hosts_final'] > 0)].copy()
vs['virus_id'] = vs['virus_id'].astype(str).str.strip()
vs = vs[~vs['virus_id'].isin(['', 'nan', 'None'])].copy()

q_hosts_hi = vs['n_hosts_final'].quantile(0.67)
q_hosts_lo = vs['n_hosts_final'].quantile(0.33)

# Build one consistent A/B/C classifier for all viruses
all_virus_categories = vs[['virus_id', 'group', 'n_hosts_final', 'n_year_batches', 'pd_final', 'pd_per_host', 'max_delta_frac']].copy()
all_virus_categories['rank_hosts'] = all_virus_categories['n_hosts_final'].rank(pct=True)
all_virus_categories['rank_pdh'] = all_virus_categories['pd_per_host'].rank(pct=True)
# Treat undefined jump fractions as least jumpy for ranking
all_virus_categories['rank_jump'] = all_virus_categories['max_delta_frac'].rank(pct=True, na_option='top')

# Raw category scores
all_virus_categories['score_A_raw'] = all_virus_categories['rank_hosts'] + (1 - all_virus_categories['rank_pdh'])
all_virus_categories['score_B_raw'] = (1 - all_virus_categories['rank_hosts']) + all_virus_categories['rank_jump']
all_virus_categories['score_C_raw'] = all_virus_categories['rank_hosts'] + all_virus_categories['rank_pdh']

# Gated scores preserve class semantics:
# A/C = high-host archetypes; B = low-host archetype
all_virus_categories['score_A'] = all_virus_categories['score_A_raw']
all_virus_categories['score_B'] = all_virus_categories['score_B_raw']
all_virus_categories['score_C'] = all_virus_categories['score_C_raw']

low_host = all_virus_categories['n_hosts_final'] <= q_hosts_lo
high_host = all_virus_categories['n_hosts_final'] >= q_hosts_hi
all_virus_categories.loc[low_host, ['score_A', 'score_C']] = -np.inf
all_virus_categories.loc[high_host, ['score_B']] = -np.inf

score_cols = ['score_A', 'score_B', 'score_C']
all_virus_categories['category_key'] = all_virus_categories[score_cols].idxmax(axis=1)

bad = np.isneginf(all_virus_categories[score_cols].max(axis=1))
if bad.any():
    raw_cols = ['score_A_raw', 'score_B_raw', 'score_C_raw']
    all_virus_categories.loc[bad, 'category_key'] = all_virus_categories.loc[bad, raw_cols].idxmax(axis=1).str.replace('_raw', '', regex=False)

all_virus_categories['category'] = all_virus_categories['category_key'].map({
    'score_A': 'A: high hosts + low PD gain',
    'score_B': 'B: low hosts + high PD jumps',
    'score_C': 'C: high hosts + high PD gain',
})

# Exemplar selection prioritizes trajectories with enough depth for informative facets
candidate_exemplars = all_virus_categories[
    (all_virus_categories['n_hosts_final'] >= MIN_EXEMPLAR_HOSTS) &
    (all_virus_categories['n_year_batches'] >= MIN_EXEMPLAR_YEAR_BATCHES)
].copy()

selected = []

def take_unique(df, n):
    out = []
    for v in df['virus_id']:
        if v not in selected and len(out) < n:
            out.append(v)
            selected.append(v)
    return out

cat_A = candidate_exemplars[candidate_exemplars['category'] == 'A: high hosts + low PD gain'].sort_values('score_A', ascending=False)
cat_B = candidate_exemplars[candidate_exemplars['category'] == 'B: low hosts + high PD jumps'].sort_values('score_B', ascending=False)
cat_C = candidate_exemplars[candidate_exemplars['category'] == 'C: high hosts + high PD gain'].sort_values('score_C', ascending=False)

A_pick = take_unique(cat_A, 2)
B_pick = take_unique(cat_B, 2)
C_pick = take_unique(cat_C, 2)

# Backfill from the depth-qualified pool if any category is underrepresented
if len(selected) < N_EXEMPLARS:
    fallback_pool = candidate_exemplars.copy()
    fallback_pool['info_score'] = (
        fallback_pool['n_year_batches'].rank(pct=True) +
        fallback_pool['n_hosts_final'].rank(pct=True) +
        fallback_pool['pd_final'].rank(pct=True)
    )
    fallback_pool = fallback_pool.sort_values(['info_score', 'n_year_batches', 'n_hosts_final'], ascending=False)
    for v in fallback_pool['virus_id']:
        if v not in selected:
            selected.append(v)
        if len(selected) >= N_EXEMPLARS:
            break

if len(selected) < N_EXEMPLARS:
    sparse_safe = all_virus_categories[(all_virus_categories['n_hosts_final'] > 2) | (all_virus_categories['n_year_batches'] > 2)]
    sparse_safe = sparse_safe.sort_values(['n_year_batches', 'n_hosts_final', 'pd_final'], ascending=False)
    for v in sparse_safe['virus_id']:
        if v not in selected:
            selected.append(v)
        if len(selected) >= N_EXEMPLARS:
            break

if len(selected) < N_EXEMPLARS:
    for v in all_virus_categories.sort_values(['n_year_batches', 'n_hosts_final', 'pd_final'], ascending=False)['virus_id']:
        if v not in selected:
            selected.append(v)
        if len(selected) >= N_EXEMPLARS:
            break

exemplars = selected[:N_EXEMPLARS]

category_map = all_virus_categories.set_index('virus_id')['category']
exemplar_table = virus_summary[virus_summary['virus_id'].isin(exemplars)].copy()
exemplar_table['category'] = exemplar_table['virus_id'].map(category_map)
exemplar_table = exemplar_table.sort_values(['category', 'n_year_batches', 'n_hosts_final'], ascending=[True, False, False]).reset_index(drop=True)
exemplar_table.to_csv(OUTPUT_DIR / 'exemplar_viruses.csv', index=False)

all_virus_categories = all_virus_categories.sort_values(['category', 'virus_id']).reset_index(drop=True)
all_virus_categories.to_csv(OUTPUT_DIR / 'all_virus_abc_categories.csv', index=False)

print('Depth filter for exemplars: hosts >=', MIN_EXEMPLAR_HOSTS, '| year batches >=', MIN_EXEMPLAR_YEAR_BATCHES)
print('Selected exemplars:', exemplars)
print('Category counts (all viruses):')
print(all_virus_categories['category'].value_counts())
exemplar_table[['virus_id','category','n_hosts_final','n_year_batches','pd_final','pd_per_host','max_delta_frac']]

In [ ]:
# Exemplar plots: PD vs year, PD vs hosts (+ permutation envelope), and ΔPD by year

def observed_hosts_in_order(v):
    d = accum_host_by_virus[v]
    if d.empty:
        return []
    return d.sort_values('host_idx')['host_tip'].tolist()


def perm_envelope_for_virus(v, r=10000, seed=1):
    hosts = observed_hosts_in_order(v)
    k = len(hosts)
    if k == 0:
        return None
    obs_curve = accum_host_by_virus[v].sort_values('host_idx')['pd_cum'].to_numpy(dtype=float)
    perms = permutation_pd_curves(hosts, r=r, seed=seed)
    return {
        'x': np.arange(1, k+1),
        'obs': obs_curve,
        'mean': np.nanmean(perms, axis=0),
        'lo': np.nanpercentile(perms, 2.5, axis=0),
        'hi': np.nanpercentile(perms, 97.5, axis=0),
    }


if 'virus_color' not in globals():
    try:
        import plotly.express as px
        _virus_ids = sorted(v for v in accum_year_by_virus.keys() if not accum_year_by_virus[v].empty)
        _palette = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
        virus_color = {v: _palette[i % len(_palette)] for i, v in enumerate(_virus_ids)}
    except Exception:
        virus_color = {}

n = len(exemplars)
fig, axes = plt.subplots(n, 3, figsize=(16, max(3*n, 8)), constrained_layout=True)
if n == 1:
    axes = np.array([axes])

perm_color = 'dimgrey'

for i, v in enumerate(exemplars):
    ay = accum_year_by_virus[v]
    v_color = virus_color.get(v, '#2f4858')

    axes[i, 0].plot(ay['year'], ay['pd_cum'], '-o', ms=3, lw=1.6, color=v_color)
    axes[i, 0].set_title(v, loc='left', fontsize=10)
    axes[i, 0].set_xlabel('Year')
    axes[i, 0].set_ylabel('Cumulative PD')

    env = perm_envelope_for_virus(v, r=R_PERM, seed=GLOBAL_SEED)

    # Year-change guides across cumulative host discovery axis
    ahv = accum_host_by_virus[v].sort_values('host_idx')[['host_idx', 'year']].copy()
    if len(ahv) > 1:
        yseq = ahv['year'].to_numpy(dtype=int)
        change_idx = np.where(np.diff(yseq) != 0)[0]
        for cidx in change_idx:
            x_sep = float(ahv['host_idx'].iloc[cidx + 1]) - 0.5
            axes[i, 1].axvline(x_sep, color='lightgrey', lw=0.8, alpha=0.8, zorder=1)

    axes[i, 1].fill_between(
        env['x'], env['lo'], env['hi'],
        facecolor='none',
        edgecolor=perm_color,
        hatch='///',
        linewidth=0.0,
        zorder=5,
        label='Permutation 95% interval' if i == 0 else None,
    )
    axes[i, 1].plot(env['x'], env['lo'], color=perm_color, lw=1.6, zorder=6)
    axes[i, 1].plot(env['x'], env['hi'], color=perm_color, lw=1.6, zorder=6)
    axes[i, 1].plot(env['x'], env['mean'], '--', color=perm_color, lw=1.4, zorder=7,
                    label='Permutation mean' if i == 0 else None)

    axes[i, 1].plot(env['x'], env['obs'], '-', color='white', lw=4.8, zorder=19)
    axes[i, 1].plot(env['x'], env['obs'], '-', color=v_color, lw=2.2, zorder=20,
                    label='Observed' if i == 0 else None)
    axes[i, 1].scatter(env['x'], env['obs'], s=34, facecolor='white', edgecolor='none', zorder=21)
    axes[i, 1].scatter(env['x'], env['obs'], s=18, facecolor=v_color, edgecolor='none', zorder=22)

    axes[i, 1].set_title(v, loc='left', fontsize=10)
    axes[i, 1].set_xlabel('Cumulative hosts')
    axes[i, 1].set_ylabel('Cumulative PD')
    axes[i, 1].grid(axis='y', ls='--', color='lightgrey', zorder=0)
    axes[i, 1].spines['top'].set_visible(False)
    axes[i, 1].spines['right'].set_visible(False)

    if i == 0:
        axes[i, 1].legend(frameon=False, fontsize=8)

    axes[i, 2].vlines(ay['year'], 0, ay['delta_pd'], lw=2, color=v_color)
    axes[i, 2].scatter(ay['year'], ay['delta_pd'], s=18, color=v_color)
    axes[i, 2].set_title(v, loc='left', fontsize=10)
    axes[i, 2].set_xlabel('Year')
    axes[i, 2].set_ylabel('ΔPD')

column_titles = ['PD vs year', 'PD vs cumulative hosts', 'ΔPD per year batch']
fig.canvas.draw()
for j, title in enumerate(column_titles):
    pos = axes[0, j].get_position()
    x_center = (pos.x0 + pos.x1) / 2
    y_top = pos.y1 + 0.015
    fig.text(x_center, y_top, title, ha='center', va='bottom', fontsize=12, fontweight='bold')

for fmt in SAVE_FORMATS:
    fig.savefig(OUTPUT_DIR / f'exemplar_pd_panels.{fmt}', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Arena vs Hanta 4-panel summary: year-colored ΔPD scatter + pooled-host cumulative PD curves
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

mpl.rcParams['font.weight'] = 300
mpl.rcParams['axes.labelweight'] = 300
mpl.rcParams['axes.labelsize'] = 16
mpl.rcParams['font.size'] = 12
mpl.rcParams['pdf.fonttype'] = 42

hanta_link_color = "#9aa8b6"
arena_link_color = "#e0b193"
log_fit_color = 'indianred'


def greyed_color(hex_color, frac_to_grey=0.55):
    """Blend a color toward neutral grey for envelope styling."""
    c = np.array(mpl.colors.to_rgb(hex_color), dtype=float)
    g = np.array([0.5, 0.5, 0.5], dtype=float)
    out = (1.0 - frac_to_grey) * c + frac_to_grey * g
    return tuple(out.tolist())


valid_groups = virus_summary['group'].dropna().astype(str).str.strip()
has_group = valid_groups.isin(['Arena', 'Hanta']).any() and valid_groups.nunique() >= 2

if not has_group:
    print('No explicit Arena/Hanta grouping available for robust comparison.')
    print('Set VIRUS_GROUP_MAP_PATH to a TSV with columns: virus_id, group (Arena/Hanta), then rerun from standardization cell.')
else:
    grp = virus_summary[virus_summary['group'].isin(['Arena','Hanta'])].copy()
    group_link = {'Arena': arena_link_color, 'Hanta': hanta_link_color}

    try:
        from scipy.stats import spearmanr
    except Exception:
        spearmanr = None

    def one_tailed_from_two_sided(rho, p_two, alternative='less'):
        if not np.isfinite(rho) or not np.isfinite(p_two):
            return np.nan
        if alternative == 'less':
            return p_two / 2 if rho < 0 else 1 - p_two / 2
        if alternative == 'greater':
            return p_two / 2 if rho > 0 else 1 - p_two / 2
        return p_two

    fig = plt.figure(figsize=(15.2, 9.2), constrained_layout=True)
    gs = fig.add_gridspec(2, 2)

    axA = fig.add_subplot(gs[0, 0])
    axB = fig.add_subplot(gs[0, 1], sharey=axA)
    axC = fig.add_subplot(gs[1, 0])
    axD = fig.add_subplot(gs[1, 1])

    scatter_axes = {'Arena': axA, 'Hanta': axB}
    curve_axes = {'Arena': axC, 'Hanta': axD}

    all_years = []
    for v in grp['virus_id'].tolist():
        ay = accum_year_by_virus.get(v)
        if ay is not None and not ay.empty:
            all_years.extend(ay['year'].dropna().astype(int).tolist())
    y_min = min(all_years) if len(all_years) else 1900
    y_max = max(all_years) if len(all_years) else 2025
    norm = mpl.colors.Normalize(vmin=y_min, vmax=y_max)
    cmap = mpl.cm.get_cmap('RdPu')

    for g in ['Arena', 'Hanta']:
        ax = scatter_axes[g]
        g_viruses = grp.loc[grp['group'] == g, 'virus_id'].tolist()
        chunks = []

        for v in g_viruses:
            ay = accum_year_by_virus.get(v)
            if ay is None or ay.empty:
                continue
            d = ay[['year', 'delta_pd']].dropna().copy()
            if d.empty:
                continue
            d['virus_id'] = v
            chunks.append(d)

        if len(chunks) == 0:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center', va='center')
            ax.set_xlabel('Year')
            ax.set_ylabel('ΔPD (per year batch)')
            ax.set_ylim(bottom=-0.05)
            continue

        g_df = pd.concat(chunks, ignore_index=True)

        ax.scatter(
            g_df['year'], g_df['delta_pd'],
            c=g_df['year'], cmap=cmap, norm=norm,
            s=72, edgecolor='none', alpha=0.9, zorder=4, clip_on=False,
        )

        max_idx = g_df.groupby('year')['delta_pd'].idxmax()
        max_df = g_df.loc[max_idx].copy()
        ax.scatter(
            max_df['year'], max_df['delta_pd'],
            c=max_df['year'], cmap=cmap, norm=norm,
            s=138, edgecolor='black', linewidth=1.8, alpha=0.98, zorder=6, clip_on=False,
        )

        if spearmanr is not None:
            if len(max_df) > 2:
                r_max, p_max_two = spearmanr(max_df['year'], max_df['delta_pd'], nan_policy='omit')
                p_max_one = one_tailed_from_two_sided(r_max, p_max_two, alternative='greater')
            else:
                r_max, p_max_one = np.nan, np.nan

            if len(g_df) > 2:
                r_all, p_all_two = spearmanr(g_df['year'], g_df['delta_pd'], nan_policy='omit')
                p_all_one = one_tailed_from_two_sided(r_all, p_all_two, alternative='less')
            else:
                r_all, p_all_one = np.nan, np.nan

            ax.text(
                0.5, 0.98,
                f"Spearman's r(max): {r_max:.2f}, p1: {p_max_one:.3f}\n"
                f"Spearman's r(all): {r_all:.2f}, p1: {p_all_one:.3f}",
                transform=ax.transAxes,
                ha='left', va='top', fontsize=9,
            )

        y_top = float(np.nanmax(g_df['delta_pd'])) if len(g_df) else 1.0
        y_pad = max(1e-6, y_top * 0.03)
        ax.set_xlabel('Year')
        ax.grid(ls='--', color='lightgrey', zorder=0)
        ax.tick_params(axis='x', rotation=45)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.set_ylim(-y_pad, max(1e-6, y_top * 1.08))

    axA.set_ylabel('ΔPD')

    cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=[axA, axB], shrink=0.88, pad=0.02)
    cbar.set_label('Year')

    if 'GROUP_PD_PERM_CACHE' not in globals():
        GROUP_PD_PERM_CACHE = {}

    for g in ['Arena', 'Hanta']:
        ax = curve_axes[g]
        link_color = group_link[g]
        env_color = greyed_color(link_color, frac_to_grey=0.60)

        g_sub = (
            vh_first.loc[vh_first['group'] == g, ['host_tip_resolved', 'first_year']]
            .dropna()
            .drop_duplicates()
            .groupby('host_tip_resolved', as_index=False)['first_year']
            .min()
            .sort_values(['first_year', 'host_tip_resolved'])
        )

        hosts_ordered = g_sub['host_tip_resolved'].astype(str).tolist()
        years_ordered = g_sub['first_year'].astype(int).to_numpy()
        k = len(hosts_ordered)

        if k < 2:
            ax.text(0.5, 0.5, 'No usable trajectories', transform=ax.transAxes, ha='center', va='center')
            ax.set_xlabel('Discovered hosts (cumulative)')
            ax.set_ylim(bottom=-0.05)
            ax.grid(axis='y', ls='--', color='lightgrey', zorder=0)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_linewidth(1.5)
            ax.spines['bottom'].set_linewidth(1.5)
            continue

        x = np.arange(1, k + 1, dtype=float)

        seen = set()
        y_obs = np.zeros(k, dtype=float)
        for i_h, h in enumerate(hosts_ordered):
            seen.add(h)
            y_obs[i_h] = faith_pd(seen)

        perm_key = (g, tuple(hosts_ordered), int(R_PERM), int(GLOBAL_SEED))
        if perm_key in GROUP_PD_PERM_CACHE:
            perms = GROUP_PD_PERM_CACHE[perm_key]
        else:
            perms = permutation_pd_curves(hosts_ordered, r=R_PERM, seed=GLOBAL_SEED)
            GROUP_PD_PERM_CACHE[perm_key] = perms

        lo = np.nanpercentile(perms, 2.5, axis=0)
        hi = np.nanpercentile(perms, 97.5, axis=0)

        ax.plot(x, lo, color=env_color, lw=1.6, zorder=6)
        ax.plot(x, hi, color=env_color, lw=1.6, zorder=6)
        ax.fill_between(
            x, lo, hi,
            facecolor='none',
            edgecolor=env_color,
            hatch='///',
            linewidth=0.0,
            zorder=5,
        )

        ax.plot(x, y_obs, color=link_color, lw=2.1, alpha=0.45, zorder=18)
        ax.scatter(x, y_obs, s=16, facecolor=link_color, edgecolor='none', zorder=20)

        # Log least-squares fit.
        X = np.column_stack([np.ones_like(x), np.log(x)])
        beta, *_ = np.linalg.lstsq(X, y_obs, rcond=None)
        y_fit = beta[0] + beta[1] * np.log(x)
        ax.plot(x, y_fit, color='white', lw=4.2, zorder=21)
        ax.plot(x, y_fit, color=log_fit_color, lw=2.3, zorder=22)

        if len(years_ordered) > 0:
            change_idx = np.where(np.diff(years_ordered) != 0)[0]
            boundaries = np.r_[0, change_idx + 1, k]
            shade = False
            for b0, b1 in zip(boundaries[:-1], boundaries[1:]):
                if shade:
                    ax.axvspan(b0 + 0.5, b1 + 0.5, facecolor='k', edgecolor='none', alpha=0.04, zorder=0)
                shade = not shade

        y_top = np.nanmax(np.r_[y_obs, hi, y_fit])
        y_pad = max(1e-6, y_top * 0.03)
        ax.set_xlabel('Discovered hosts (cumulative)')
        ax.grid(axis='y', ls='--', color='lightgrey', zorder=0)
        ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, nbins=10))
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.set_ylim(-y_pad, max(1e-6, y_top * 1.06))

        legend_handles = [
            Patch(facecolor='none', edgecolor=env_color, hatch='///', label='Permutation 95% interval'),
            Line2D([0], [0], color=link_color, lw=2.1, alpha=0.45,
                   marker='o', markersize=5, markerfacecolor=link_color,
                   markeredgecolor='none', label='Observed'),
            Line2D([0], [0], color=log_fit_color, lw=2.3, label='Log least-squares fit'),
        ]
        ax.legend(handles=legend_handles, frameon=False, fontsize=10, loc='lower right')

    axC.set_ylabel('Cumulative PD contribution')
    axD.set_ylabel('Cumulative PD contribution')

    for ax, letter in zip([axA, axB, axC, axD], ['A.', 'B.', 'C.', 'D.']):
        ax.text(0.01, 0.98, letter, transform=ax.transAxes, ha='left', va='top', fontsize=16, fontweight='bold')

    for ax in [axA, axB, axC, axD]:
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)

    for fmt in SAVE_FORMATS:
        fig.savefig(OUTPUT_DIR / f'group_pd_4panel.{fmt}', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Predict hosts remaining until PD gain becomes negligible (from log least-squares fit)
# Model used in panel curves: PD(x) = a + b*log(x), so marginal gain per additional host is dPD/dx = b/x.
# Here we evaluate epsilon as a fraction of current fitted marginal gain and plot projections for
# multipliers 0.75, 0.5, and 0.25 in faceted (Arena/Hanta) panels.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EPS_MODE = 'current_gain_per_host_fit'  # options: current_gain_per_host_fit, observed_min_delta_pd, upham_min_branch, manual
CURRENT_GAIN_MULTIPLIER = 1.0           # baseline epsilon for main summary table
PROJECTION_MULTIPLIERS = [0.75, 0.5, 0.25]  # requested forecast scenarios
PD_NEGLIGIBLE_EPS_MANUAL = 1.0
MIN_EPS_FLOOR = 1e-6
EPS_MULTIPLIERS = [2.0, 1.0, 0.5, 0.2]


def get_upham_min_positive_branch_length(tree):
    vals = []
    for node in tree.preorder_node_iter():
        if node is tree.seed_node:
            continue
        bl = node.edge.length
        if bl is None:
            continue
        bl = float(bl)
        if np.isfinite(bl) and bl > 0:
            vals.append(bl)
    return float(np.min(vals)) if len(vals) else np.nan


def choose_epsilon(eps_mode, y_obs, upham_min, b_hat, k_obs):
    if eps_mode == 'current_gain_per_host_fit':
        if k_obs > 0 and np.isfinite(b_hat):
            gain_now = float(b_hat / k_obs)
            if np.isfinite(gain_now) and gain_now > 0:
                eps = max(gain_now * float(CURRENT_GAIN_MULTIPLIER), MIN_EPS_FLOOR)
                return eps, f'current fitted gain per host * {CURRENT_GAIN_MULTIPLIER:g}'

    if eps_mode == 'observed_min_delta_pd':
        d_obs = np.diff(np.r_[0.0, y_obs])
        pos = d_obs[np.isfinite(d_obs) & (d_obs > 0)]
        if len(pos):
            eps = max(float(np.min(pos)), MIN_EPS_FLOOR)
            return eps, 'observed minimum positive ΔPD (group)'

    if eps_mode == 'upham_min_branch':
        if np.isfinite(upham_min) and upham_min > 0:
            return max(float(upham_min), MIN_EPS_FLOOR), 'Upham tree minimum positive branch length'

    return max(float(PD_NEGLIGIBLE_EPS_MANUAL), MIN_EPS_FLOOR), 'manual fallback'


def summarize_for_eps(group_name, eps, eps_source, upham_min_pd, a_hat, b_hat, k_obs):
    if b_hat <= 0 or eps <= 0:
        x_negl = float(k_obs)
    else:
        x_negl = max(float(k_obs), b_hat / float(eps))

    hosts_left = int(max(0.0, np.ceil(x_negl) - k_obs))
    pd_now = float(a_hat + b_hat * np.log(k_obs))
    pd_negl = float(a_hat + b_hat * np.log(max(1.0, x_negl)))
    delta_remain = max(0.0, pd_negl - pd_now)
    gain_now = float(b_hat / k_obs) if k_obs > 0 else np.nan

    return {
        'group': group_name,
        'epsilon_pd_per_host': float(eps),
        'epsilon_source': eps_source,
        'eps_mode': EPS_MODE,
        'current_gain_multiplier': float(CURRENT_GAIN_MULTIPLIER),
        'upham_min_positive_branch_length': upham_min_pd,
        'n_hosts_observed': int(k_obs),
        'b_log_slope': b_hat,
        'current_gain_per_host_fit': gain_now,
        'x_negligible_fit': x_negl,
        'hosts_left_to_negligible_fit': hosts_left,
        'pd_fit_current': pd_now,
        'pd_fit_at_negligible': pd_negl,
        'pd_fit_remaining_until_negligible': delta_remain,
    }


upham_min_pd = get_upham_min_positive_branch_length(host_tree)
print(f'EPS_MODE: {EPS_MODE}')
if EPS_MODE == 'current_gain_per_host_fit':
    print(f'CURRENT_GAIN_MULTIPLIER: {CURRENT_GAIN_MULTIPLIER:g}')
    if abs(float(CURRENT_GAIN_MULTIPLIER) - 1.0) < 1e-12:
        print('Note: multiplier=1 implies x_negligible = n_hosts_observed (hosts_left ~ 0 by definition).')
if np.isfinite(upham_min_pd):
    print(f'Upham min positive branch length: {upham_min_pd:.6g}')
else:
    print('Upham min positive branch length: NA')

valid_groups = virus_summary['group'].dropna().astype(str).str.strip()
has_group = valid_groups.isin(['Arena', 'Hanta']).any() and valid_groups.nunique() >= 2

if not has_group:
    print('No explicit Arena/Hanta grouping available for robust prediction.')
else:
    rows_main = []
    rows_sens = []
    rows_proj = []
    fit_cache = {}

    for g in ['Arena', 'Hanta']:
        g_sub = (
            vh_first.loc[vh_first['group'] == g, ['host_tip_resolved', 'first_year']]
            .dropna()
            .drop_duplicates()
            .groupby('host_tip_resolved', as_index=False)['first_year']
            .min()
            .sort_values(['first_year', 'host_tip_resolved'])
        )

        hosts_ordered = g_sub['host_tip_resolved'].astype(str).tolist()
        k_obs = len(hosts_ordered)
        if k_obs < 3:
            continue

        x = np.arange(1, k_obs + 1, dtype=float)
        seen = set()
        y_obs = np.zeros(k_obs, dtype=float)
        for i_h, h in enumerate(hosts_ordered):
            seen.add(h)
            y_obs[i_h] = faith_pd(seen)

        X = np.column_stack([np.ones_like(x), np.log(x)])
        beta, *_ = np.linalg.lstsq(X, y_obs, rcond=None)
        a_hat, b_hat = float(beta[0]), float(beta[1])
        fit_cache[g] = {'a_hat': a_hat, 'b_hat': b_hat, 'k_obs': k_obs, 'y_obs': y_obs}

        eps_group, eps_source = choose_epsilon(EPS_MODE, y_obs, upham_min_pd, b_hat, k_obs)
        eps_group = max(float(eps_group), MIN_EPS_FLOOR)
        eps_sensitivity = [eps_group * float(m) for m in EPS_MULTIPLIERS]

        rows_main.append(summarize_for_eps(g, eps_group, eps_source, upham_min_pd, a_hat, b_hat, k_obs))
        for eps in eps_sensitivity:
            rows_sens.append(summarize_for_eps(g, eps, eps_source, upham_min_pd, a_hat, b_hat, k_obs))

        # Requested scenario projections relative to current gain
        gain_now = float(b_hat / k_obs)
        for m in PROJECTION_MULTIPLIERS:
            eps_proj = max(gain_now * float(m), MIN_EPS_FLOOR)
            s = summarize_for_eps(
                g, eps_proj,
                f'current fitted gain per host * {m:g}',
                upham_min_pd,
                a_hat, b_hat, k_obs,
            )
            s['projection_multiplier'] = float(m)
            rows_proj.append(s)

    pred_df = pd.DataFrame(rows_main).sort_values('group').reset_index(drop=True)
    sens_df = pd.DataFrame(rows_sens).sort_values(['group', 'epsilon_pd_per_host']).reset_index(drop=True)
    proj_df = pd.DataFrame(rows_proj).sort_values(['group', 'projection_multiplier']).reset_index(drop=True)

    display(pred_df)
    print('Sensitivity across epsilon thresholds (multiples of group-specific epsilon):')
    display(sens_df)
    print('Projection scenarios requested (multipliers of current fitted gain):')
    display(proj_df[['group', 'projection_multiplier', 'epsilon_pd_per_host', 'x_negligible_fit', 'hosts_left_to_negligible_fit', 'pd_fit_remaining_until_negligible']])

    pred_path = OUTPUT_DIR / 'group_hosts_left_until_negligible.csv'
    sens_path = OUTPUT_DIR / 'group_hosts_left_until_negligible_sensitivity.csv'
    proj_path = OUTPUT_DIR / 'group_hosts_left_until_negligible_current_gain_multipliers.csv'
    pred_df.to_csv(pred_path, index=False)
    sens_df.to_csv(sens_path, index=False)
    proj_df.to_csv(proj_path, index=False)
    print('Saved:', pred_path)
    print('Saved:', sens_path)
    print('Saved:', proj_path)

    # Faceted projection curves for multipliers 0.75, 0.5, 0.25
    fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.4), constrained_layout=True, sharey=False)
    colors = {0.75: '#4c78a8', 0.5: '#f58518', 0.25: '#54a24b'}

    for ax, g in zip(axes, ['Arena', 'Hanta']):
        if g not in fit_cache:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center', va='center')
            continue

        a_hat = fit_cache[g]['a_hat']
        b_hat = fit_cache[g]['b_hat']
        k_obs = fit_cache[g]['k_obs']
        y_obs = fit_cache[g]['y_obs']
        x_obs = np.arange(1, k_obs + 1, dtype=float)

        # Observed + fitted over observed range
        ax.scatter(x_obs, y_obs, s=14, color='black', zorder=5, label='Observed')
        y_fit_obs = a_hat + b_hat * np.log(x_obs)
        ax.plot(x_obs, y_fit_obs, color='indianred', lw=1.8, zorder=6, label='Log fit (observed range)')

        g_proj = proj_df[proj_df['group'] == g].copy()
        for m in PROJECTION_MULTIPLIERS:
            row = g_proj[g_proj['projection_multiplier'].round(6) == round(float(m), 6)]
            if row.empty:
                continue
            row = row.iloc[0]
            x_negl = float(row['x_negligible_fit'])
            x_end = int(np.ceil(max(k_obs, x_negl)))
            x_ext = np.arange(k_obs, max(k_obs + 1, x_end + 1), dtype=float)
            y_ext = a_hat + b_hat * np.log(x_ext)

            ax.plot(x_ext, y_ext, lw=2.0, color=colors[m],
                    label=f'm={m:g} (hosts left={int(row["hosts_left_to_negligible_fit"])})')
            ax.axvline(x_negl, color=colors[m], lw=1.2, ls='--', alpha=0.9)

        ax.set_title(g)
        ax.set_xlabel('Hosts (cumulative)')
        ax.set_ylabel('Cumulative PD (fit/observed)')
        ax.grid(ls='--', color='lightgrey', zorder=0)
        ax.set_ylim(bottom=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[1].legend(frameon=False, fontsize=8, loc='lower right')
    for fmt in SAVE_FORMATS:
        fig.savefig(OUTPUT_DIR / f'group_pd_projection_current_gain_multipliers.{fmt}', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
print('Output directory:', OUTPUT_DIR)
print('Key files:')
for name in [
    'virus_summary.csv',
    'per_virus_pd_by_year.csv',
    'per_virus_pd_by_hosts.csv',
    'exemplar_viruses.csv',
    'all_virus_abc_categories.csv',
    'f_late_sensitivity.csv',
    'unmatched_host_labels.tsv',
]:
    p = OUTPUT_DIR / name
    print(' -', p, '| exists:', p.exists())

print('Figures exported as:', SAVE_FORMATS)